# 01 - TF-IDF Similarity Experiments

This notebook explores how TF-IDF + cosine similarity scores the sample resumes against the two sample job descriptions, and sanity-checks that relevant resumes score higher than irrelevant ones.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import pandas as pd
from api.scoring import compute_tfidf_similarity
from api.parsing import normalize_text

DATA_DIR = Path('..') / 'data'
JD_DIR = DATA_DIR / 'sample_jds'
RESUME_DIR = DATA_DIR / 'sample_resumes'
print(list(JD_DIR.glob('*.txt')))
print(list(RESUME_DIR.glob('*.txt')))

[WindowsPath('../data/sample_jds/backend_engineer.txt'), WindowsPath('../data/sample_jds/data_analyst.txt')]
[WindowsPath('../data/sample_resumes/aisha_rahman_ml_engineer.txt'), WindowsPath('../data/sample_resumes/carlos_mendez_sales_weak_fit.txt'), WindowsPath('../data/sample_resumes/daniel_osei_data_analyst_junior.txt'), WindowsPath('../data/sample_resumes/james_okafor_backend_senior.txt'), WindowsPath('../data/sample_resumes/lena_kowalski_backend_mid.txt'), WindowsPath('../data/sample_resumes/maria_gonzalez_data_analyst_mid.txt'), WindowsPath('../data/sample_resumes/priya_sharma_data_analyst_senior.txt'), WindowsPath('../data/sample_resumes/sophia_lindqvist_data_engineer.txt'), WindowsPath('../data/sample_resumes/tom_becker_backend_junior.txt')]


In [2]:
jd_text = normalize_text((JD_DIR / 'data_analyst.txt').read_text(encoding='utf-8'))
resume_files = sorted(RESUME_DIR.glob('*.txt'))
resume_texts = [normalize_text(f.read_text(encoding='utf-8')) for f in resume_files]
resume_names = [f.name for f in resume_files]
len(resume_texts)

9

In [3]:
sims = compute_tfidf_similarity(jd_text, resume_texts)
df = pd.DataFrame({'resume': resume_names, 'similarity': sims})
df = df.sort_values('similarity', ascending=False).reset_index(drop=True)
df

,resume,similarity
0,priya_sharma_data_analyst_senior.txt,0.438544
1,daniel_osei_data_analyst_junior.txt,0.357484
2,maria_gonzalez_data_analyst_mid.txt,0.306994
3,sophia_lindqvist_data_engineer.txt,0.270445
4,aisha_rahman_ml_engineer.txt,0.125137
5,tom_becker_backend_junior.txt,0.064710
6,lena_kowalski_backend_mid.txt,0.050796
7,james_okafor_backend_senior.txt,0.041608
8,carlos_mendez_sales_weak_fit.txt,0.040894


## Observation

Resumes explicitly written for data-analyst-style roles (Priya, Maria, Daniel) should score meaningfully higher on TF-IDF similarity against the Data Analyst JD than the sales or backend-only resumes, even before we add skill/experience/education signal on top.

In [4]:
jd_text_be = normalize_text((JD_DIR / 'backend_engineer.txt').read_text(encoding='utf-8'))
sims_be = compute_tfidf_similarity(jd_text_be, resume_texts)
df_be = pd.DataFrame({'resume': resume_names, 'similarity': sims_be})
df_be = df_be.sort_values('similarity', ascending=False).reset_index(drop=True)
df_be

,resume,similarity
0,james_okafor_backend_senior.txt,0.438263
1,lena_kowalski_backend_mid.txt,0.318196
2,tom_becker_backend_junior.txt,0.166192
3,aisha_rahman_ml_engineer.txt,0.129335
4,sophia_lindqvist_data_engineer.txt,0.097440
5,daniel_osei_data_analyst_junior.txt,0.062533
6,priya_sharma_data_analyst_senior.txt,0.047286
7,maria_gonzalez_data_analyst_mid.txt,0.036734
8,carlos_mendez_sales_weak_fit.txt,0.025669


## Why TF-IDF instead of embeddings?

See `docs/ARCHITECTURE.md` for the full discussion. In short: TF-IDF is free, requires no model download or GPU, fits per-request in milliseconds on a serverless function, and is fully explainable (we can literally see which terms drive the score) — all valuable properties for a zero-cost, recruiter-facing tool.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_features=30)
matrix = vectorizer.fit_transform([jd_text] + resume_texts[:3])
import pandas as pd
pd.DataFrame(matrix.toarray(), columns=vectorizer.get_feature_names_out(),
             index=['JD'] + resume_names[:3]).T.sort_values('JD', ascending=False).head(15)

,JD,aisha_rahman_ml_engineer.txt,carlos_mendez_sales_weak_fit.txt,daniel_osei_data_analyst_junior.txt
data,0.599981,0.219761,0.000000,0.617794
experience,0.306577,0.119779,0.122517,0.126272
analysis,0.299990,0.073254,0.000000,0.077224
analyst,0.277911,0.000000,0.000000,0.381549
bi,0.277911,0.000000,0.000000,0.286162
sql,0.224993,0.146507,0.000000,0.231673
business,0.185274,0.000000,0.277653,0.000000
excel,0.185274,0.000000,0.000000,0.381549
power,0.185274,0.000000,0.000000,0.190775
analytics,0.185274,0.000000,0.000000,0.190775


Above: top TF-IDF terms for the JD vs. the first three resumes, illustrating why the similarity score is explainable — high-weight shared terms like 'sql', 'python', or 'tableau' directly drive the cosine similarity.